# Serving Ludwig LLMs with vLLM

This notebook shows how to deploy a Ludwig LLM using the [vLLM](https://github.com/vllm-project/vllm) backend for high-throughput, production-grade inference.

**What you will learn**
- Launch a Ludwig vLLM server from the command line
- Query the `/predict` and `/batch_predict` REST endpoints
- Use the OpenAI-compatible `/v1/completions` endpoint
- Benchmark vLLM latency vs the default FastAPI backend
- Production tips: tensor parallelism and continuous batching

> **Prerequisites:** a trained Ludwig LLM model saved on disk. The notebook uses placeholder paths; replace them with your own.

In [ ]:
!pip install "ludwig[llm,vllm]" --quiet

## Launch vLLM server

Start the Ludwig serve command with `--backend vllm`. This replaces the default FastAPI/HuggingFace inference loop with vLLM's continuous-batching engine.

```bash
ludwig serve \
    --model_path=/path/to/model \
    --backend vllm \
    --num_gpus 1
```

The server starts on `http://0.0.0.0:8000` by default. You can change the host and port with `--host` and `--port`.

**Key flags:**

| Flag | Default | Description |
|------|---------|-------------|
| `--backend vllm` | `local` | Use vLLM inference engine |
| `--num_gpus N` | `1` | Number of GPUs (enables tensor parallelism when N > 1) |
| `--host` | `0.0.0.0` | Bind address |
| `--port` | `8000` | Port |

Once the server prints `Application startup complete`, it is ready to accept requests.

In [ ]:
# In Colab, launch the server in the background via a shell cell.
# Replace /path/to/model with your actual model directory.
#
# !ludwig serve --model_path=/path/to/model --backend vllm --num_gpus 1 &
#
# Give the server a few seconds to start before running subsequent cells.
import time
SERVER_URL = "http://localhost:8000"
print(f"Server URL: {SERVER_URL}")

## Single prediction

The `/predict` endpoint accepts a single example as form data. Field names must match the input feature names used when the model was trained.

### curl

```bash
curl http://localhost:8000/predict \
    -X POST \
    -F 'text=Explain what a transformer model is in one sentence.'
```

### Python requests

In [ ]:
import json
import time

import requests

PREDICT_URL = f"{SERVER_URL}/predict"

prompt = "Explain what a transformer model is in one sentence."

t0 = time.perf_counter()
response = requests.post(PREDICT_URL, data={"text": prompt})
latency_ms = (time.perf_counter() - t0) * 1000

response.raise_for_status()
result = response.json()

print(f"Latency: {latency_ms:.1f} ms")
print(json.dumps(result, indent=2))

## Batch prediction

The `/batch_predict` endpoint accepts multiple examples encoded as a JSON string in the Pandas `split` format.

### curl

```bash
curl http://localhost:8000/batch_predict \
    -X POST \
    -F 'dataset={"columns":["text"],"data":[["What is Ludwig?"],["What is vLLM?"],["What is PagedAttention?"]]}'
```

In [ ]:
BATCH_PREDICT_URL = f"{SERVER_URL}/batch_predict"

prompts = [
    "What is Ludwig?",
    "What is vLLM?",
    "What is PagedAttention?",
    "What is tensor parallelism?",
    "Summarise continuous batching in one sentence.",
]

dataset_json = json.dumps({"columns": ["text"], "data": [[p] for p in prompts]})

t0 = time.perf_counter()
response = requests.post(BATCH_PREDICT_URL, data={"dataset": dataset_json})
elapsed = time.perf_counter() - t0

response.raise_for_status()
result = response.json()

print(f"{len(prompts)} examples in {elapsed:.2f} s ({len(prompts)/elapsed:.1f} examples/s)")
print("Columns:", result["columns"])
for i, row in enumerate(result["data"]):
    print(f"  [{i}] {row[0][:120]}")

## OpenAI-compatible endpoint

When Ludwig serves via vLLM it optionally exposes an OpenAI-compatible `/v1/completions` endpoint. This lets you use the OpenAI Python client or any tool that speaks the OpenAI API without any code changes.

Enable it with `--enable-openai-api` when starting the server:

```bash
ludwig serve --model_path=/path/to/model --backend vllm --num_gpus 1 --enable-openai-api
```

In [ ]:
OPENAI_URL = f"{SERVER_URL}/v1/completions"

payload = {
    "model": "ludwig-model",  # vLLM accepts any string here
    "prompt": "Explain retrieval-augmented generation in two sentences.",
    "max_tokens": 128,
    "temperature": 0.7,
}

response = requests.post(OPENAI_URL, json=payload)

if response.status_code == 404:
    print("OpenAI-compatible endpoint not enabled. Start the server with --enable-openai-api.")
else:
    response.raise_for_status()
    result = response.json()
    print(result["choices"][0]["text"].strip())

In [ ]:
# You can also use the openai Python package directly.
# !pip install openai --quiet

# from openai import OpenAI

# client = OpenAI(
#     base_url=f"{SERVER_URL}/v1",
#     api_key="not-required",  # vLLM does not enforce API keys by default
# )

# completion = client.completions.create(
#     model="ludwig-model",
#     prompt="Describe the attention mechanism in transformers.",
#     max_tokens=150,
# )
# print(completion.choices[0].text.strip())

## Latency comparison

The following cells time the same requests against:
1. The **default FastAPI backend** (standard HuggingFace `model.generate`)
2. The **vLLM backend** (continuous batching + PagedAttention)

The comparison requires two server instances to be running simultaneously — one on port 8000 (vLLM) and one on port 8001 (default). The cells below print results for whichever servers are reachable.

In [ ]:
import statistics

DEFAULT_URL = "http://localhost:8001"  # default FastAPI backend
VLLM_URL    = "http://localhost:8000"  # vLLM backend

BENCH_PROMPTS = [
    "What is machine learning?",
    "Explain gradient descent.",
    "What is overfitting?",
    "Describe a convolutional neural network.",
    "What is the difference between supervised and unsupervised learning?",
]


def measure_latencies(base_url: str, prompts: list, n_warmup: int = 2) -> list:
    """Return a list of per-request latencies (seconds) for the given server."""
    url = f"{base_url}/predict"
    latencies = []
    for i, prompt in enumerate(prompts * n_warmup + prompts):
        t0 = time.perf_counter()
        try:
            r = requests.post(url, data={"text": prompt}, timeout=120)
            r.raise_for_status()
        except Exception as exc:
            print(f"  Request failed: {exc}")
            continue
        elapsed = time.perf_counter() - t0
        if i >= len(prompts) * n_warmup:  # skip warmup
            latencies.append(elapsed)
    return latencies


def print_stats(label: str, latencies: list) -> None:
    if not latencies:
        print(f"{label}: no data (server not running?)")
        return
    p50 = statistics.median(latencies)
    p95 = sorted(latencies)[int(0.95 * len(latencies))]
    mean = statistics.mean(latencies)
    print(f"{label}: mean={mean*1000:.0f} ms  p50={p50*1000:.0f} ms  p95={p95*1000:.0f} ms  ({len(latencies)} requests)")


print("Benchmarking default backend (port 8001)...")
default_latencies = measure_latencies(DEFAULT_URL, BENCH_PROMPTS)
print_stats("Default backend", default_latencies)

print("\nBenchmarking vLLM backend (port 8000)...")
vllm_latencies = measure_latencies(VLLM_URL, BENCH_PROMPTS)
print_stats("vLLM backend   ", vllm_latencies)

if default_latencies and vllm_latencies:
    speedup = statistics.mean(default_latencies) / statistics.mean(vllm_latencies)
    print(f"\nvLLM speedup: {speedup:.1f}x on sequential single-request latency")

## Production tips

### Tensor parallelism

For models that don't fit in a single GPU's VRAM, vLLM can split the model across multiple GPUs using tensor parallelism. Pass `--num_gpus N` to Ludwig serve:

```bash
# Split a large model across 4 GPUs
ludwig serve \
    --model_path=/path/to/large_model \
    --backend vllm \
    --num_gpus 4
```

vLLM will automatically set the tensor parallel degree to match `--num_gpus`. Each GPU handles a shard of the weight matrices and they communicate via NVLink or PCIe during inference.

### Continuous batching

Unlike traditional static batching (where all requests in a batch must finish before a new one starts), vLLM's continuous batching inserts new requests into an ongoing batch as soon as a slot becomes free. This means:

- **GPU utilisation stays high** even when requests have very different output lengths
- **Tail latency is lower** because short requests are not held up by long ones
- **Throughput increases** roughly 2–10x vs static batching for typical LLM workloads

No configuration is needed — continuous batching is enabled automatically when you use `--backend vllm`.

### PagedAttention

vLLM's PagedAttention manages the KV cache in fixed-size pages (similar to virtual memory in an OS). This eliminates almost all memory fragmentation from the KV cache, allowing you to serve more concurrent requests from the same GPU.

### Memory and concurrency tuning

Pass vLLM-specific arguments via `--vllm-extra-args` (a JSON string):

```bash
ludwig serve \
    --model_path=/path/to/model \
    --backend vllm \
    --num_gpus 1 \
    --vllm-extra-args '{"gpu_memory_utilization": 0.9, "max_num_seqs": 256}'
```

| Parameter | Default | Description |
|-----------|---------|-------------|
| `gpu_memory_utilization` | `0.9` | Fraction of GPU memory reserved for the KV cache |
| `max_num_seqs` | `256` | Maximum concurrent sequences in a single iteration |
| `max_model_len` | model default | Maximum total sequence length (prompt + output) |

### Quantization

vLLM supports AWQ, GPTQ, and SqueezeLLM quantization for reduced memory and faster inference:

```bash
ludwig serve \
    --model_path=/path/to/awq_model \
    --backend vllm \
    --vllm-extra-args '{"quantization": "awq"}'
```

### Monitoring

Ludwig serve exposes a `/metrics` Prometheus endpoint. See the `prometheus_monitoring/` directory in this folder for a ready-to-use Docker Compose stack with Prometheus and Grafana.